In [1]:
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from roost import InferenceModelResourceManager
import torch.nn.functional as F
from tqdm.notebook import tqdm

from server.models.coref import CorefModel
from server.manuscript import Manuscript, Section

resource_manager = InferenceModelResourceManager(quota_gb=15)

SURFACE, GRID, BASELINE, MUTED, INK = "#fcfcfb", "#e1e0d9", "#c3c2b7", "#898781", "#0b0b0b"
ACCENT, REST = "#2a78d6", "#c3c2b7"
WARM, COOL = "#d64a2a", "#2a78d6"
ATTRIBUTION = LinearSegmentedColormap.from_list("attribution", [WARM, SURFACE, COOL])



In [2]:
manuscript = Manuscript.load(Path("./data/story_2.md"))
sections = manuscript.sections
lines = manuscript.lines

## Nodes

The line is not the node. A node is a thing the story refers to, and lines are the
evidence for it — which is what lets "she" and "Anabelle" be one node when their
encodings are nowhere near each other.

### Coreference resolution
Graph nodes represent entities from the text - unique entities.

So if in our text we have a character, then all references to that character, either named (e.g. Anabelle) or anonymous (e.g. she, her) - have to be identified and attributed to a single node.

The ML task that does that is called "coreference resolution".
An example of a model trained for this type of task is "biu-nlp/lingmess-coref"

In [3]:
@dataclass(frozen=True)
class KnowledgeItem:
    """Represents a single entity from a body of knowledge.
    
    That thing might be a subject - a person, a place, an object; a concept; an event; a situation.
    """

    id: int  # Unique id
    mentions: tuple[tuple[int, int], ...]  # where in text is the item mentioned (start character, end character)
    name: str = ""  # what it is called, where a person annotated it rather than a model

@dataclass(frozen=True)
class KnowledgeRelation:
    start: int
    end: int

@dataclass(frozen=True)
class KnowledgeGraph:
    nodes: list[KnowledgeItem]
    edges: list[KnowledgeRelation]


### Extract the nods from the script

In [4]:
STORIES = manuscript.sections[:2]
COREF_MODEL = "biu-nlp/lingmess-coref"
COREF_MODEL_GB = 2

coref_model = CorefModel(COREF_MODEL, resource_manager, COREF_MODEL_GB)

def _story_text(section: Section):
    """A section as the resolver reads it, and as the annotation was written against."""
    return "\n".join(section.lines)


def detect_knowledge_graph_nodes(section: Section) -> list[KnowledgeItem]:
    return [
        KnowledgeItem(id=index, mentions=tuple(map(tuple, cluster)))
        for index, cluster in enumerate(coref_model.clusters(_story_text(section)))
    ]

items_of_story = [
    detect_knowledge_graph_nodes(section)
    for section in tqdm(STORIES, desc="resolving", unit="story")
]

# Out
#   items_of_story  one tuple of KnowledgeItem per story, in that story's own coordinates


resolving:   0%|          | 0/2 [00:00<?, ?story/s]

14:11:20  31760 resource_manager Preparing biu-nlp/lingmess-coref for serving ...
14:11:22  31760 httpx        HTTP Request: HEAD https://huggingface.co/biu-nlp/lingmess-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:11:22  31760 httpx        HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/lingmess-coref/fa5d8a827a09388d03adbe9e800c7d8c509c3935/config.json "HTTP/1.1 200 OK"
14:11:22  31760 httpx        HTTP Request: HEAD https://huggingface.co/biu-nlp/lingmess-coref/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
14:11:22  31760 httpx        HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/biu-nlp/lingmess-coref/fa5d8a827a09388d03adbe9e800c7d8c509c3935/config.json "HTTP/1.1 200 OK"
14:11:22  31760 httpx        HTTP Request: HEAD https://huggingface.co/biu-nlp/lingmess-coref/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
14:11:22  31760 httpx        HTTP Request: HEAD https://huggingface.c

Loading weights: 100%|██████████| 565/565 [00:00<00:00, 85644.44it/s]
[transformers] LingMessModel LOAD REPORT from: biu-nlp/lingmess-coref
Key                                | Status     |  | 
-----------------------------------+------------+--+-
longformer.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


14:11:23  31760 httpx        HTTP Request: GET https://huggingface.co/api/models/biu-nlp/lingmess-coref/commits/main "HTTP/1.1 200 OK"
14:11:23  31760 fastcoref.modeling ***** Running Inference on 1 texts *****


Map: 100%|██████████| 1/1 [00:00<00:00, 59.03 examples/s]
/Users/piotrtrochim/GitHub/writer/.venv/lib/python3.12/site-packages/pyarrow/compute.py:230: FutureWarning: Specifying null_placement in SortOptions is deprecated as of 25.0.0. Specify null_placement per sort_key instead.
  return options_class(*args, **kwargs)
Inference:   0%|          | 0/1 [00:00<?, ?it/s]

...

Inference: 100%|██████████| 1/1 [00:00<00:00,  1.50it/s]


14:11:24  31760 fastcoref.modeling Tokenize 1 inputs...


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

14:11:24  31760 fastcoref.modeling ***** Running Inference on 1 texts *****


Inference: 100%|██████████| 1/1 [00:01<00:00,  1.13s/it]


### Discover the relationships between the detected entities

Our task is to discover relationships between two entities given the following constraints:
- those two entities appear multiple times in the text
- they may be related by many different relationships
- a single may, but doesn't have to, connect them
- we don't know the names of those relationships.

There doesn't exist a single algorithm that would solve that problem. Here's what I found to exist.

#### OpenRE

OpenRE is the standard for discovering relations between entities when the set of
possible relations is not known in advance. In practice it is done at sentence level:
one sentence, one entity pair, one relation.
The sentence in this case is a real, linguistic sentence, punctuation delimited.

It outputs k unlabeled clusters that encode relations between the entities.

#### ATLOP
Adaptive Thresholding and Localized cOntext Pooling, introduced in [Document-Level Relation Extraction with Adaptive Thresholding and Localized Context Pooling](https://arxiv.org/abs/2010.11304) in 2021.

Tick nearly every box on our list, except form the last one - it requires a fixed list of named relations — its output layer has one logit per relation, and nothing outside that list can be emitted.

#### Schema induction

The state of the art is [AutoSchemaKG](https://arxiv.org/html/2505.23628v1) which uses an LLM (so an LLM decoder).

It's a pipeline (not a model) that involves an intruction-tuned LLM prompted to discover entities and relationships between them. It produces named relationships, that we could certainly benefit from. However - it uses a generative approach, meaning that the model decides WHAT relationships to generate for us - and will return a lot of False Negatives (missing relationships) and False Positives (some named relationships may be mislabeled or outright connect incorrect entities)



In [5]:
from collections import Counter

from server.models.atlop import AtlopModel

ATLOP_MODEL = "roberta-large"
ATLOP_MODEL_GB = 3

atlop_model = AtlopModel(ATLOP_MODEL, resource_manager, ATLOP_MODEL_GB)


def detect_knowledge_graph_edges(
    section: Section, items: list[KnowledgeItem]
) -> list[KnowledgeRelation]:
    """Every relation an ordered pair of nodes clears its own threshold on, one edge each.

    A pair related in more than one way comes back as that many edges between the same
    two nodes. Which relation each of them stands for is DocRED's to say, and is not
    something a `KnowledgeRelation` carries, so what survives is how many.
    """
    return [
        KnowledgeRelation(items[head].id, items[tail].id)
        for head, tail, _ in atlop_model.relations(
            _story_text(section), [item.mentions for item in items]
        )
    ]


edges_of_story = [
    detect_knowledge_graph_edges(section, items)
    for section, items in zip(tqdm(STORIES, desc="relating", unit="story"), items_of_story)
]

for section, edges in zip(STORIES, edges_of_story):
    ways = Counter((edge.start, edge.end) for edge in edges)
    print(f"\n{section.title} — {len(edges)} edges over {len(ways)} pairs")
    for (start, end), count in ways.most_common(12):
        print(f"  {start:>3} → {end:<3}  related {count} ways")

# Out
#   edges_of_story  one list of KnowledgeRelation per story, repeated where a pair is
#                   related in more than one way, in that story's own node ids


relating:   0%|          | 0/2 [00:00<?, ?story/s]

14:11:27  31771 resource_manager Preparing roberta-large for serving ...
14:11:27  31771 httpx        HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"


Loading weights: 100%|██████████| 389/389 [00:00<00:00, 10038.17it/s]
[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


14:11:31  31771 httpx        HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/config.json "HTTP/1.1 200 OK"
14:11:31  31771 httpx        HTTP Request: HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
14:11:31  31771 httpx        HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
14:11:31  31771 httpx        HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
14:11:31  31771 httpx        HTTP Request: GET https://huggingface.co/api/models/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
14:11:31  31771 httpx        HTTP Request: GET https://huggingface.co/api/models/FacebookAI/roberta-large/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
14:11:31  31771

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (750 > 512). Running this sequence through the model will result in indexing errors



Preface — 0 edges over 0 pairs

One — 0 edges over 0 pairs


## Grading

In [6]:
preface = manuscript.sections[0]
chapter_1 = manuscript.sections[1]
stories = [preface, chapter_1]

@dataclass(frozen=True)
class KnowledgeRelation:
    start: int
    end: int

@dataclass(frozen=True)
class KnowledgeGraph:
    nodes: list[KnowledgeItem]
    edges: list[KnowledgeRelation]

relationships_in_story_1 = KnowledgeGraph(
    nodes=[
        KnowledgeItem(0, name="World", mentions=(
            preface.reference(line=0, phrase="different reality"),
            preface.reference(line=1, phrase="world of abundance, beauty, and peace"),
            preface.reference(line=1, phrase="that reality"),
            preface.reference(line=2, phrase="the outer lands"),
            preface.reference(line=4, phrase="This society"),
        )),
        KnowledgeItem(1, name="Men", mentions=(
            preface.reference(line=1, phrase="For men"),
            preface.reference(line=2, phrase="a man"),
            preface.reference(line=2, phrase="He can walk"),
            preface.reference(line=2, phrase="he can enter"),
            preface.reference(line=2, phrase="he must accept"),
            preface.reference(line=2, phrase="men serve"),
            preface.reference(line=3, phrase="The men who surrender"),
            preface.reference(line=3, phrase="Drones"),
            preface.reference(line=3, phrase="Companions"),
            preface.reference(line=4, phrase="a man"),
            preface.reference(line=4, phrase="a subject"),
        )),
        KnowledgeItem(2, name="Women", mentions=(
            preface.reference(line=1, phrase="the women"),
            preface.reference(line=2, phrase="Women rule"),
            preface.reference(line=3, phrase="her every command"),
            preface.reference(line=3, phrase="bow before her"),
            preface.reference(line=3, phrase="Mistresses"),
            preface.reference(line=4, phrase="the ruling class"),
            preface.reference(line=4, phrase="no woman"),
            preface.reference(line=4, phrase="a partner"),
            preface.reference(line=5, phrase="a woman"),
            preface.reference(line=5, phrase="she"),
        )),
        KnowledgeItem(3, name="Outside the matriarchy", mentions=(
            preface.reference(line=2, phrase="outer lands"),
            preface.reference(line=2, phrase="patriarchal society"),
        )),
        KnowledgeItem(3, name="Matriarchy", mentions=(
            preface.reference(line=1, phrase="world of abundance, beauty, and peace"),
            preface.reference(line=2, phrase="Matriarchy"),
            preface.reference(line=3, phrase="the Society"),
            preface.reference(line=4, phrase="This society"),
        )),
        KnowledgeItem(4, name="Freedom", mentions=(
            preface.reference(line=1, phrase="freedom"),
            preface.reference(line=2, phrase="walk away"),
            preface.reference(line=2, phrase="a life of humble poverty"),
            preface.reference(line=2, phrase="Women rule"),
            preface.reference(line=3, phrase="sovereignty"),
        )),
        KnowledgeItem(5, name="Servitude", mentions=(
            preface.reference(line=1, phrase="safety"),
            preface.reference(line=2, phrase="enter the Matriarchy"),
            preface.reference(line=2, phrase="enveloped in safety"),
            preface.reference(line=2, phrase="men serve"),
            preface.reference(line=3, phrase="Disobedience"),
            preface.reference(line=3, phrase="punishments"),
            preface.reference(line=3, phrase="The men who surrender"),
            preface.reference(line=3, phrase="devotion"),
            preface.reference(line=3, phrase="personal maids"),
            preface.reference(line=3, phrase="bound"),
            preface.reference(line=4, phrase="strict rules that govern it"),
            preface.reference(line=4, phrase="controlling basic male instincts"),
        )),
    ],
    edges=[
        KnowledgeRelation(1, 0),  # Men -> World
        KnowledgeRelation(2, 0),  # Women -> World
        KnowledgeRelation(1, 3),  # Men -> Outside the matriarchy
        KnowledgeRelation(1, 4),  # Men -> Matriarchy
        KnowledgeRelation(2, 4),  # Women -> Matriarchy
        KnowledgeRelation(3, 0),  # Outside the matriarchy -> World
        KnowledgeRelation(4, 0),  # Matriarchy -> World
        KnowledgeRelation(1, 4),  # Men -> Freedom
        KnowledgeRelation(1, 5),  # Men -> Servitude
        KnowledgeRelation(2, 4),  # Women -> Freedom
    ],
)

relationships_in_story_2 = KnowledgeGraph(
    nodes=[
        KnowledgeItem(0, name="Anabelle", mentions=(
            chapter_1.reference(line=0, phrase="Anabelle"),
            chapter_1.reference(line=0, phrase="She stretched"),
            chapter_1.reference(line=1, phrase="Her"),
            chapter_1.reference(line=1, phrase="She"),
            chapter_1.reference(line=1, phrase="her stomach"),
            chapter_1.reference(line=2, phrase="She"),
            chapter_1.reference(line=2, phrase="a single fingertip"),
            chapter_1.reference(line=3, phrase="She bit into"),
            chapter_1.reference(line=4, phrase="she liked"),
            chapter_1.reference(line=4, phrase="her"),
            chapter_1.reference(line=5, phrase="Anabelle"),
            chapter_1.reference(line=5, phrase="She loved"),
            chapter_1.reference(line=5, phrase="she washed"),
            chapter_1.reference(line=6, phrase="she finished"),
            chapter_1.reference(line=6, phrase="she swung her feet"),
            chapter_1.reference(line=6, phrase="her toes"),
            chapter_1.reference(line=6, phrase="She stepped"),
            chapter_1.reference(line=7, phrase="She"),
            chapter_1.reference(line=8, phrase="she twirled"),
            chapter_1.reference(line=8, phrase="her heel"),
            chapter_1.reference(line=8, phrase="She was meeting"),
            chapter_1.reference(line=9, phrase="her legs"),
            chapter_1.reference(line=9, phrase="where she wanted"),
            chapter_1.reference(line=10, phrase="She draped"),
            chapter_1.reference(line=10, phrase="she pressed"),
            chapter_1.reference(line=11, phrase="before her"),
            chapter_1.reference(line=11, phrase="her thighs"),
            chapter_1.reference(line=11, phrase="her face"),
            chapter_1.reference(line=11, phrase="she might have caught"),
            chapter_1.reference(line=11, phrase="she had"),
            chapter_1.reference(line=12, phrase="her feet"),
            chapter_1.reference(line=12, phrase="her instep"),
            chapter_1.reference(line=13, phrase="She checked"),
            chapter_1.reference(line=13, phrase="her reflection"),
        )),
        KnowledgeItem(1, name="Companion", mentions=(
            chapter_1.reference(line=1, phrase="Companion"),
            chapter_1.reference(line=2, phrase="he strained"),
            chapter_1.reference(line=3, phrase="Her Companion"),
            chapter_1.reference(line=4, phrase="keep him"),
            chapter_1.reference(line=4, phrase="a Companion"),
            chapter_1.reference(line=4, phrase="He held"),
            chapter_1.reference(line=4, phrase="he had"),
            chapter_1.reference(line=4, phrase="he first"),
            chapter_1.reference(line=5, phrase="his hip"),
            chapter_1.reference(line=5, phrase="his skin"),
            chapter_1.reference(line=5, phrase="on him"),
            chapter_1.reference(line=6, phrase="her Companion"),
            chapter_1.reference(line=6, phrase="his mouth"),
            chapter_1.reference(line=10, phrase="Her Companion"),
            chapter_1.reference(line=11, phrase="He stretched"),
            chapter_1.reference(line=11, phrase="his stiff arms and knelt"),
            chapter_1.reference(line=11, phrase="his hands"),
            chapter_1.reference(line=11, phrase="His eyes"),
            chapter_1.reference(line=11, phrase="his gaze"),
            chapter_1.reference(line=11, phrase="challenge him"),
            chapter_1.reference(line=12, phrase="Her Companion"),
            chapter_1.reference(line=12, phrase="His arms"),
            chapter_1.reference(line=12, phrase="his wrists"),
        )),
        KnowledgeItem(2, name="Sophia", mentions=(
            chapter_1.reference(line=8, phrase="Sophia"),
            chapter_1.reference(line=9, phrase="Sophia's gaze"),
        )),
        KnowledgeItem(3, name="Anabelle's outfit", mentions=(
            chapter_1.reference(line=6, phrase="her slippers"),
            chapter_1.reference(line=8, phrase="something appropriate"),
            chapter_1.reference(line=8, phrase="something appropriate"),
            chapter_1.reference(line=8, phrase="sexy, yet professional and understated"),
            chapter_1.reference(line=9, phrase="nothing suitable"),
            chapter_1.reference(line=9, phrase="an outfit for the morning"),
            chapter_1.reference(line=9, phrase="form-fitting black sequined dress"),
            chapter_1.reference(line=9, phrase="blazer"),
            chapter_1.reference(line=9, phrase="a jade necklace on a fine gold chain"),
            chapter_1.reference(line=10, phrase="the clothes"),
            chapter_1.reference(line=11, phrase="Panties"),
            chapter_1.reference(line=11, phrase="hosiery"),
            chapter_1.reference(line=11, phrase="skirt"),
            chapter_1.reference(line=11, phrase="dress"),
            chapter_1.reference(line=12, phrase="stilettos"),
        )),
        KnowledgeItem(4, name="Companion's outfit", mentions=(
            chapter_1.reference(line=1, phrase="his chastity belt"),
            chapter_1.reference(line=2, phrase="the little bell attached to his caged genitals"),
            chapter_1.reference(line=4, phrase="The chastity belt"),
            chapter_1.reference(line=4, phrase="he small gauge set into the metal"),
            chapter_1.reference(line=4, phrase="her collar"),
            chapter_1.reference(line=10, phrase="the release button on his collar"),
            chapter_1.reference(line=10, phrase="the armbinder"),
            chapter_1.reference(line=12, phrase="the armbinder"),
            chapter_1.reference(line=13, phrase="leather leash"),
            chapter_1.reference(line=13, phrase="the ring on his collar"),
        )),
        KnowledgeItem(5, name="Bedroom", mentions=(
            chapter_1.reference(line=6, phrase="the edge of the mattress"),
            chapter_1.reference(line=6, phrase="bed"),
            chapter_1.reference(line=6, phrase="the window"),
            chapter_1.reference(line=8, phrase="the wardrobe"),
            chapter_1.reference(line=10, phrase="the bed"),
            chapter_1.reference(line=13, phrase="hook by the door"),
        )),
        KnowledgeItem(6, name="Shopping trip & day plans", mentions=(
            chapter_1.reference(line=7, phrase="a day she had long anticipated"),
            chapter_1.reference(line=7, phrase="an offer from a prestigious law firm"),
            chapter_1.reference(line=8, phrase="reckless shopping spree"),
            chapter_1.reference(line=8, phrase="proper hot date"),
            chapter_1.reference(line=9, phrase="the shopping trip"),
        )),
        KnowledgeItem(7, name="Breakfast", mentions=(
            chapter_1.reference(line=1, phrase="the glass and took a long sip"),
            chapter_1.reference(line=1, phrase="the sweet, silky blend of carrot and apple"),
            chapter_1.reference(line=1, phrase="the crispy, golden-brown crust of the croissant"),
            chapter_1.reference(line=3, phrase="the delicious pastry"),
            chapter_1.reference(line=5, phrase="the pastry"),
            chapter_1.reference(line=5, phrase="solid swig of juice"),
        )),
        KnowledgeItem(8, name="Gestures", mentions=(
            chapter_1.reference(line=3, phrase="a lazy, whirlpool gesture in the air with her other hand"),
            chapter_1.reference(line=10, phrase="flicked her fingers"),
        )),
        KnowledgeItem(9, name="Poses", mentions=(
            chapter_1.reference(line=3, phrase="turning around"),
            chapter_1.reference(line=4, phrase="the pose"),
        )),
        KnowledgeItem(10, name="Routines", mentions=(
            chapter_1.reference(line=4, phrase="his milking"),
            chapter_1.reference(line=11, phrase="a routine trained deep into his muscles"),
            chapter_1.reference(line=12, phrase="planted a reverent kiss on her instep, signaling his task was complete"),
        )),
    ],
    edges=[
        KnowledgeRelation(1, 0),  # Companion -> Anabelle
        KnowledgeRelation(0, 2),  # Anabelle -> Sophia
        KnowledgeRelation(0, 3),  # Anabelle -> Anabelle's outfit
        KnowledgeRelation(1, 3),  # Companion -> Anabelle's outfit
        KnowledgeRelation(0, 4),  # Anabelle -> Companion's outfit
        KnowledgeRelation(1, 4),  # Companion -> Companion's outfit
        KnowledgeRelation(0, 5),  # Anabelle -> Bedroom
        KnowledgeRelation(1, 5),  # Companion -> Bedroom
        KnowledgeRelation(1, 5),  # Companion -> Bedroom
        KnowledgeRelation(0, 6),  # Anabelle -> Shopping trip & day plans
        KnowledgeRelation(0, 7),  # Anabelle -> Breakfast
        KnowledgeRelation(1, 7),  # Companion -> Breakfast
        KnowledgeRelation(0, 8),  # Anabelle -> Gestures
        KnowledgeRelation(8, 1),  # Gestures -> Companion
        KnowledgeRelation(8, 9),  # Gestures -> Poses
        KnowledgeRelation(1, 9),  # Companion -> Poses
        KnowledgeRelation(0, 10),  # Anabelle -> Routines
        KnowledgeRelation(1, 10),  # Companion -> Routines
    ],
)

relationships_in_story_1

KnowledgeGraph(nodes=[KnowledgeItem(id=0, mentions=((32, 48), (8, 44), (113, 124), (75, 89), (0, 11)), name='World'), KnowledgeItem(id=1, mentions=((104, 110), (35, 39), (55, 65), (191, 202), (290, 303), (335, 343), (237, 257), (304, 309), (346, 355), (282, 286), (305, 313)), name='Men'), KnowledgeItem(id=2, mentions=((81, 89), (323, 332), (123, 139), (146, 159), (401, 410), (182, 197), (259, 266), (322, 330), (21, 27), (103, 105)), name='Women'), KnowledgeItem(id=3, mentions=((79, 89), (131, 149)), name='Outside the matriarchy'), KnowledgeItem(id=3, mentions=((8, 44), (208, 217), (10, 20), (0, 11)), name='Matriarchy'), KnowledgeItem(id=4, mentions=((159, 165), (62, 70), (101, 124), (323, 332), (51, 61)), name='Freedom'), KnowledgeItem(id=5, mentions=((170, 175), (198, 217), (220, 238), (335, 343), (179, 190), (214, 224), (237, 257), (293, 300), (370, 383), (386, 390), (55, 81), (113, 144)), name='Servitude')], edges=[KnowledgeRelation(start=1, end=0), KnowledgeRelation(start=2, end=0)

In [7]:
from collections import Counter
from typing import NamedTuple

ANNOTATED = (relationships_in_story_1, relationships_in_story_2)

Span = tuple[int, int]


class Tally(NamedTuple):
    """Every annotated mention falls in exactly one of these."""

    correct: int  # found, and grouped with the rest of its own entity
    wrong: int  # found, but grouped with some other entity
    missed: int  # no found mention covers it

    @property
    def total(self) -> int:
        return self.correct + self.wrong + self.missed


class EntityTally(NamedTuple):
    name: str
    mentions: int
    correct: int
    resolved_as: int | None  # the found entity most of its mentions were grouped into


def overlap(lhs: Span, rhs: Span) -> int:
    """How many characters two half-open spans hold in common."""
    return max(0, min(lhs[1], rhs[1]) - max(lhs[0], rhs[0]))


def filed_under(mention: Span, found: tuple[KnowledgeItem, ...]) -> int | None:
    """The found entity whose own mention covers this one most, if any covers it.

    Boundaries are drawn by hand on one side and by a model on the other, so the two
    mark the same mention without marking the same characters; the widest overlap is
    taken as the answer rather than an exact match, which would find almost nothing.
    """
    filed, covered = None, 0
    for item in found:
        for span in item.mentions:
            shared = overlap(mention, span)
            if shared > covered:
                filed, covered = item.id, shared
    return filed


def called(item: KnowledgeItem, text: str) -> str:
    """How the story first puts a found entity, which reads where its number does not."""
    start, end = min(item.mentions)
    return text[start:end]


def tallied(
    annotated: KnowledgeGraph, found: tuple[KnowledgeItem, ...]
) -> tuple[Tally, list[EntityTally]]:
    """How much of each annotated entity one found entity accounts for.

    The resolver never saw the annotation, so which found entity answers for which
    annotated one has to be inferred: it is whichever one most of that entity's
    mentions were grouped into, and its remaining mentions are counted against it.
    """
    filed = {
        mention: filed_under(mention, found)
        for item in annotated.nodes
        for mention in item.mentions
    }

    per_entity = []
    for index, item in enumerate(annotated.nodes):
        grouped = Counter(
            filed[mention] for mention in item.mentions if filed[mention] is not None
        )
        prevailing = (
            min(grouped, key=lambda entity: (-grouped[entity], entity))
            if grouped
            else None
        )
        per_entity.append(
            EntityTally(
                name=item.name or f"entity {index}",
                mentions=len(item.mentions),
                correct=grouped[prevailing] if prevailing is not None else 0,
                resolved_as=prevailing,
            )
        )

    total = sum(len(item.mentions) for item in annotated.nodes)
    placed = sum(
        1
        for item in annotated.nodes
        for mention in item.mentions
        if filed[mention] is not None
    )
    correct = sum(entity.correct for entity in per_entity)
    return Tally(correct, placed - correct, total - placed), per_entity


for section, annotated, found in zip(STORIES, ANNOTATED, items_of_story):
    text = _story_text(section)
    resolved = {item.id: called(item, text) for item in found}
    tally, per_entity = tallied(annotated, found)

    print(f"\n{section.title} — {tally.total} annotated mentions")
    print(f"  correct  {tally.correct:>4}  {tally.correct / tally.total:>4.0%}")
    print(f"  wrong    {tally.wrong:>4}  {tally.wrong / tally.total:>4.0%}")
    print(f"  missed   {tally.missed:>4}  {tally.missed / tally.total:>4.0%}")
    print("\n  what I called it            of its mentions  the resolver grouped them as")
    for entity in per_entity:
        share = entity.correct / entity.mentions
        grouped_as = (
            "nothing — none of them were found"
            if entity.resolved_as is None
            else f'"{resolved[entity.resolved_as][:36]}"'
        )
        print(
            f"  {entity.name:<26}"
            f"  {entity.correct:>3}/{entity.mentions:<3} {share:>4.0%}"
            f"     {grouped_as}"
        )



Preface — 49 annotated mentions
  correct    12   24%
  wrong       6   12%
  missed     31   63%

  what I called it            of its mentions  the resolver grouped them as
  World                         2/5    40%     "a different reality"
  Men                           2/11   18%     "a singular choice: freedom or safety"
  Women                         1/10   10%     "a different reality"
  Outside the matriarchy        0/2     0%     nothing — none of them were found
  Matriarchy                    1/4    25%     "a different reality"
  Freedom                       2/5    40%     "a different reality"
  Servitude                     4/12   33%     "a singular choice: freedom or safety"

One — 108 annotated mentions
  correct    42   39%
  wrong      16   15%
  missed     50   46%

  what I called it            of its mentions  the resolver grouped them as
  Anabelle                     14/34   41%     "Anabelle"
  Companion                     8/23   35%     "Anabelle"
  Soph

TODO:
- the edge creation
- a benchmark for testing the accuracy of the search
- the search using the graph
- extracting plots
- extracting characters